In [1]:
# ============================================================
# FULL PART 6 — Run this single cell
# ============================================================
!pip install wbdata linearmodels plotly -q

import wbdata
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from linearmodels.panel import PanelOLS

# EU-27
eu27 = {
    'AUT': 'Austria', 'BEL': 'Belgium', 'BGR': 'Bulgaria', 'HRV': 'Croatia',
    'CYP': 'Cyprus', 'CZE': 'Czechia', 'DNK': 'Denmark', 'EST': 'Estonia',
    'FIN': 'Finland', 'FRA': 'France', 'DEU': 'Germany', 'GRC': 'Greece',
    'HUN': 'Hungary', 'IRL': 'Ireland', 'ITA': 'Italy', 'LVA': 'Latvia',
    'LTU': 'Lithuania', 'LUX': 'Luxembourg', 'MLT': 'Malta', 'NLD': 'Netherlands',
    'POL': 'Poland', 'PRT': 'Portugal', 'ROU': 'Romania', 'SVK': 'Slovakia',
    'SVN': 'Slovenia', 'ESP': 'Spain', 'SWE': 'Sweden'
}

indicators = {
    'NE.EXP.GNFS.ZS': 'exports_gdp_pct',
    'NE.IMP.GNFS.ZS': 'imports_gdp_pct',
}

df = wbdata.get_dataframe(indicators, country=list(eu27.keys()), date=("2015", "2023"))
df = df.reset_index()
df['country_name'] = df['country'].map(lambda x: eu27.get(x, x))
df['trade_balance_pct_gdp'] = df['exports_gdp_pct'] - df['imports_gdp_pct']

rvi_scores = {
    'Czechia': 100.0, 'Ireland': 89.4, 'Slovakia': 87.0, 'Hungary': 80.6,
    'Romania': 80.5, 'Slovenia': 79.2, 'Germany': 70.9, 'Poland': 63.0,
    'Finland': 62.5, 'Lithuania': 59.6, 'Estonia': 59.3, 'Austria': 59.3,
    'Italy': 54.6, 'Bulgaria': 54.2, 'Sweden': 49.7, 'Denmark': 42.4,
    'Belgium': 39.8, 'Spain': 39.8, 'Malta': 39.7, 'Croatia': 36.2,
    'Netherlands': 33.8, 'Portugal': 33.1, 'Latvia': 29.0, 'France': 24.5,
    'Greece': 18.0, 'Luxembourg': 0.4, 'Cyprus': 0.0
}

df['RVI'] = df['country_name'].map(rvi_scores)
df['date'] = df['date'].astype(int)
df['Post2020'] = (df['date'] >= 2020).astype(int)
df['RVI_x_Post2020'] = df['RVI'] * df['Post2020']

df_panel = df.dropna(subset=['trade_balance_pct_gdp', 'RVI']).copy()
df_panel = df_panel.drop(columns=['country'])
df_panel = df_panel.set_index(['country_name', 'date'])

print("Data ready! Shape:", df_panel.shape)

# ── CHART 1: Scatter RVI vs Trade Balance Change ──
df_panel_reset = df_panel.reset_index()
pre  = df_panel_reset[df_panel_reset['date'] < 2020].groupby('country_name')['trade_balance_pct_gdp'].mean()
post = df_panel_reset[df_panel_reset['date'] >= 2020].groupby('country_name')['trade_balance_pct_gdp'].mean()
tb_change = (post - pre).reset_index()
tb_change.columns = ['country_name', 'tb_change']
tb_change['RVI'] = tb_change['country_name'].map(rvi_scores)
tb_change = tb_change.dropna()

fig1 = px.scatter(
    tb_change, x='RVI', y='tb_change', text='country_name', trendline='ols',
    labels={'RVI': 'Reshoring Vulnerability Index (0–100)',
            'tb_change': 'Change in Trade Balance (% GDP, Post-2020 vs Pre-2020)'},
    title='RVI vs Trade Balance Change Post-2020 — EU-27',
    color_discrete_sequence=['#2E86AB']
)
fig1.update_traces(textposition='top center', marker=dict(size=8))
fig1.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.5)
fig1.update_layout(plot_bgcolor='white', paper_bgcolor='white', font=dict(family='Arial', size=13))
fig1.write_html('fig_part6_scatter.html')
fig1.show()
print("Chart 1 done!")

# ── CHART 2: Pre vs Post Bar sorted by RVI ──
tb_merged = tb_change.sort_values('RVI', ascending=False)
fig2 = go.Figure()
fig2.add_trace(go.Bar(name='Pre-2020 avg', x=tb_merged['country_name'],
    y=pre[tb_merged['country_name']].values, marker_color='#A8DADC'))
fig2.add_trace(go.Bar(name='Post-2020 avg', x=tb_merged['country_name'],
    y=post[tb_merged['country_name']].values, marker_color='#E63946'))
fig2.update_layout(
    barmode='group',
    title='Trade Balance (% GDP): Pre vs Post 2020 — Sorted by RVI (High → Low)',
    xaxis_title='Country (sorted by Reshoring Vulnerability)',
    yaxis_title='Trade Balance (% GDP)',
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=12), xaxis_tickangle=-45,
    legend=dict(orientation='h', y=1.1)
)
fig2.write_html('fig_part6_bar.html')
fig2.show()
print("Chart 2 done!")

# ── CHART 3: Trade Balance Change by Country ──
tb_merged2 = tb_change.sort_values('tb_change')
fig3 = px.bar(
    tb_merged2, x='country_name', y='tb_change',
    color='tb_change', color_continuous_scale=['#E63946', '#F4A261', '#2A9D8F'],
    labels={'country_name': 'Country', 'tb_change': 'Trade Balance Change (% GDP)'},
    title='Post-2020 Trade Balance Change by Country — EU-27<br><sup>Positive = improved | Negative = worsened</sup>'
)
fig3.add_hline(y=0, line_dash='dash', line_color='black', opacity=0.4)
fig3.update_layout(
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=12), xaxis_tickangle=-45,
    coloraxis_showscale=False
)
fig3.write_html('fig_part6_change.html')
fig3.show()
print("Chart 3 done!")

print("\nAll 3 charts saved!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.5/300.5 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 8.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
Data ready! Shape: (234, 6)


Chart 1 done!


Chart 2 done!


Chart 3 done!

All 3 charts saved!


In [2]:
from google.colab import files
files.download('fig_part6_scatter.html')
files.download('fig_part6_bar.html')
files.download('fig_part6_change.html')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>